# Logging Lab — Meghana

Python's built-in `logging` library records what an application is doing so you can debug issues and monitor behavior. This notebook works through the basics from the starter notebook, fixes a few problems in it, and then applies logging to a small ML training loop, which is how it's used in MLOps pipelines.

Everything here uses only the Python standard library.

### 1. Importing the Logging Library

In [1]:
import logging
import os

# Write all log files into a logs/ folder next to this notebook
LOG_DIR = 'logs'
os.makedirs(LOG_DIR, exist_ok=True)

### 2. Basic Logging Configuration
`basicConfig` sets up the root logger. It only takes effect the **first** time it's called, unless you pass `force=True`. Jupyter sometimes configures logging before your code runs, so `force=True` makes sure this configuration is the one actually used.

In [2]:
FORMAT = '%(asctime)s - %(name)s - %(levelname)s - %(message)s'

logging.basicConfig(level=logging.DEBUG, format=FORMAT, force=True)

### 3. Logging Messages
There are five standard levels, from least to most severe: **DEBUG**, **INFO**, **WARNING**, **ERROR**, **CRITICAL**. Setting a level shows messages at that level *and above*, so with `level=DEBUG` all five appear.

In [3]:
logging.debug("This is a debug message")        # Detailed information for debugging
logging.info("This is an info message")          # General information
logging.warning("This is a warning message")     # A potential issue
logging.error("This is an error message")        # An error in the application
logging.critical("This is a critical message")   # A serious error

2026-11-15 18:06:07,189 - root - DEBUG - This is a debug message
2026-11-15 18:06:07,190 - root - INFO - This is an info message
2026-11-15 18:06:07,190 - root - WARNING - This is a warning message
2026-11-15 18:06:07,190 - root - ERROR - This is an error message
2026-11-15 18:06:07,191 - root - CRITICAL - This is a critical message


### 4. Using Loggers
Rather than logging through the root logger, create named loggers for different parts of an application. The name shows up in every message, and each logger's settings can be controlled separately.

In [4]:
logger = logging.getLogger("my_module")

logger.debug("Debug message from my_module")
logger.info("Info message from my_module")

2026-11-15 18:06:07,194 - my_module - DEBUG - Debug message from my_module
2026-11-15 18:06:07,195 - my_module - INFO - Info message from my_module


### 5. Logging Exceptions
`logger.exception()` logs at ERROR level and automatically includes the traceback. It should be called inside an `except` block.

In [5]:
try:
    result = 10 / 0
except ZeroDivisionError:
    logging.exception("An error occurred while dividing by zero")

2026-11-15 18:06:07,199 - root - ERROR - An error occurred while dividing by zero
Traceback (most recent call last):
  File "/var/folders/6s/tkkbfmw52494xt6qfyl36sxm0000gn/T/ipykernel_68891/2034576387.py", line 2, in <module>
    result = 10 / 0
             ~~~^~~
ZeroDivisionError: division by zero


### 6. Logging to a File
**Fix from the starter notebook:** calling `basicConfig(filename='app.log', ...)` a second time silently does nothing, because the root logger was already configured in step 2. In the starter, `app.log` was never written by that cell. Adding `force=True` replaces the earlier configuration so messages really go to the file.

In [6]:
APP_LOG = os.path.join(LOG_DIR, 'app.log')

logging.basicConfig(filename=APP_LOG, level=logging.DEBUG, format=FORMAT, force=True)

logging.info("This message goes to app.log, not the notebook output")
logging.warning("So does this one")

# Read the file back to show it worked
with open(APP_LOG) as f:
    print(f.read())

2026-11-15 18:06:07,203 - root - INFO - This message goes to app.log, not the notebook output
2026-11-15 18:06:07,203 - root - WARNING - So does this one



### 7. Controlling Log Levels
The level can be changed at runtime. After setting the root logger to WARNING, DEBUG and INFO messages are filtered out.

In [7]:
# Send output back to the notebook for the rest of the examples
logging.basicConfig(level=logging.DEBUG, format=FORMAT, force=True)

logging.getLogger().setLevel(logging.WARNING)

logging.info("You will NOT see this (INFO is below WARNING)")
logging.warning("You WILL see this")

# Reset for the next sections
logging.getLogger().setLevel(logging.DEBUG)

2026-11-15 18:06:07,206 - root - WARNING - You WILL see this


### 8. Log Handlers
Handlers decide **where** messages go. A logger can have several, each with its own level and format.

**Fixes from the starter notebook:**
- The starter's output printed every message **twice**: once from the logger's own handler and again because the message also *propagated* up to the root logger. Setting `propagate = False` stops that.
- The starter's handlers had no formatter, so one copy appeared without a timestamp or level. Each handler now gets a formatter.
- Running the starter's handler cell again would add the same handlers a second time (and duplicate output again). Clearing `logger.handlers` first makes the cell safe to re-run.
- Fixed the typo "noccurred" in the message.

Here, the console only shows INFO and above, while the file keeps everything, including DEBUG. That's a common setup: a clean console plus a detailed file for debugging later.

In [8]:
logger = logging.getLogger("my_module")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()        # safe to re-run this cell
logger.propagate = False       # don't also send messages to the root logger

formatter = logging.Formatter(FORMAT)

console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(formatter)

MODULE_LOG = os.path.join(LOG_DIR, 'my_module.log')
file_handler = logging.FileHandler(MODULE_LOG, mode='w')
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(formatter)

logger.addHandler(console_handler)
logger.addHandler(file_handler)

In [9]:
logger.debug("Only in the file (DEBUG is below the console's INFO level)")
logger.info("In the console and the file")

try:
    result = 10 / 0
except ZeroDivisionError:
    logger.exception("An error occurred while dividing by zero")

2026-11-15 18:06:07,215 - my_module - INFO - In the console and the file
2026-11-15 18:06:07,216 - my_module - ERROR - An error occurred while dividing by zero
Traceback (most recent call last):
  File "/var/folders/6s/tkkbfmw52494xt6qfyl36sxm0000gn/T/ipykernel_68891/277438769.py", line 5, in <module>
    result = 10 / 0
             ~~~^~~
ZeroDivisionError: division by zero


In [10]:
# The file has the DEBUG line the console skipped
print(open(MODULE_LOG).read())

2026-11-15 18:06:07,215 - my_module - DEBUG - Only in the file (DEBUG is below the console's INFO level)
2026-11-15 18:06:07,215 - my_module - INFO - In the console and the file
2026-11-15 18:06:07,216 - my_module - ERROR - An error occurred while dividing by zero
Traceback (most recent call last):
  File "/var/folders/6s/tkkbfmw52494xt6qfyl36sxm0000gn/T/ipykernel_68891/277438769.py", line 5, in <module>
    result = 10 / 0
             ~~~^~~
ZeroDivisionError: division by zero



## My extensions

### 9. Rotating log files
A single log file grows forever in a long-running service. `RotatingFileHandler` starts a new file once the current one reaches a size limit and keeps a fixed number of old ones (`rotating.log.1`, `rotating.log.2`, …).

In [11]:
from logging.handlers import RotatingFileHandler

rot_logger = logging.getLogger("rotating_demo")
rot_logger.setLevel(logging.INFO)
rot_logger.handlers.clear()
rot_logger.propagate = False

ROT_LOG = os.path.join(LOG_DIR, 'rotating.log')
rot_handler = RotatingFileHandler(ROT_LOG, maxBytes=500, backupCount=2)  # tiny limit so rotation happens
rot_handler.setFormatter(logging.Formatter(FORMAT))
rot_logger.addHandler(rot_handler)

for i in range(30):
    rot_logger.info(f"Request {i} handled")

print(sorted(f for f in os.listdir(LOG_DIR) if f.startswith('rotating')))

['rotating.log', 'rotating.log.1', 'rotating.log.2']


### 10. Structured (JSON) logs
Plain-text logs are easy for people to read but hard for tools to search. Logging platforms (like the ELK stack covered in a later lab) work best with JSON, where each field is separate. A custom `Formatter` turns each record into one JSON line, including any extra fields passed with `extra=`.

In [12]:
import json

class JsonFormatter(logging.Formatter):
    def format(self, record):
        entry = {
            "time": self.formatTime(record),
            "logger": record.name,
            "level": record.levelname,
            "message": record.getMessage(),
        }
        # Include any extra fields passed with extra={...}
        for key in ("epoch", "loss", "lr"):
            if hasattr(record, key):
                entry[key] = getattr(record, key)
        return json.dumps(entry)

json_logger = logging.getLogger("json_demo")
json_logger.setLevel(logging.INFO)
json_logger.handlers.clear()
json_logger.propagate = False

json_handler = logging.StreamHandler()
json_handler.setFormatter(JsonFormatter())
json_logger.addHandler(json_handler)

json_logger.info("Training started", extra={"lr": 0.01})
json_logger.warning("Loss went up", extra={"epoch": 4, "loss": 0.73})

{"time": "2026-11-15 18:06:07,237", "logger": "json_demo", "level": "INFO", "message": "Training started", "lr": 0.01}
{"time": "2026-11-15 18:06:07,237", "logger": "json_demo", "level": "WARNING", "message": "Loss went up", "epoch": 4, "loss": 0.73}


### 11. Logging an ML training run
This puts it all together in an MLOps-style example: a tiny linear regression model trained with gradient descent (pure Python, no libraries), logging:

- **INFO** for run start/end and progress every 100 epochs,
- **DEBUG** for every epoch (file only),
- **WARNING** if the loss goes up, which can signal a learning rate that's too high,
- **ERROR** with a traceback if training diverges (loss above 1,000,000, NaN or infinite), which stops the run early.

The run is done twice: once with a sensible learning rate and once with one that's too high, so the warnings and error show up.

In [13]:
import math
import random

def make_train_logger(run_name):
    log = logging.getLogger(f"train.{run_name}")
    log.setLevel(logging.DEBUG)
    log.handlers.clear()
    log.propagate = False

    console = logging.StreamHandler()
    console.setLevel(logging.INFO)
    console.setFormatter(logging.Formatter(FORMAT))

    file = logging.FileHandler(os.path.join(LOG_DIR, f'train_{run_name}.log'), mode='w')
    file.setLevel(logging.DEBUG)
    file.setFormatter(logging.Formatter(FORMAT))

    log.addHandler(console)
    log.addHandler(file)
    return log


def train(run_name, lr, epochs=500):
    log = make_train_logger(run_name)

    # Synthetic data: y = 3x + 2 + noise
    random.seed(42)
    xs = [random.uniform(0, 10) for _ in range(100)]
    ys = [3 * x + 2 + random.gauss(0, 1) for x in xs]
    n = len(xs)

    w, b = 0.0, 0.0
    prev_loss = None
    log.info(f"Run started: lr={lr}, epochs={epochs}, samples={n}")

    try:
        for epoch in range(1, epochs + 1):
            preds = [w * x + b for x in xs]
            loss = sum((p - y) ** 2 for p, y in zip(preds, ys)) / n

            if math.isnan(loss) or math.isinf(loss) or loss > 1e6:
                raise ValueError(f"Training diverged at epoch {epoch}: loss={loss:.3g}")

            log.debug(f"epoch={epoch} loss={loss:.4f} w={w:.4f} b={b:.4f}")
            if prev_loss is not None and loss > prev_loss:
                log.warning(f"Loss increased at epoch {epoch}: {prev_loss:.4f} -> {loss:.4f}")
            if epoch % 100 == 0:
                log.info(f"epoch {epoch}/{epochs} loss={loss:.4f}")

            # Gradient descent step
            dw = 2 / n * sum((p - y) * x for p, y, x in zip(preds, ys, xs))
            db = 2 / n * sum(p - y for p, y in zip(preds, ys))
            w -= lr * dw
            b -= lr * db
            prev_loss = loss

        log.info(f"Run finished: w={w:.3f}, b={b:.3f} (true values: 3, 2), final loss={loss:.4f}")
    except ValueError:
        log.exception("Training failed")

In [14]:
# A good learning rate: loss goes down steadily
train("good_lr", lr=0.01)

2026-11-15 18:06:07,249 - train.good_lr - INFO - Run started: lr=0.01, epochs=500, samples=100
2026-11-15 18:06:07,255 - train.good_lr - INFO - epoch 100/500 loss=1.1221
2026-11-15 18:06:07,260 - train.good_lr - INFO - epoch 200/500 loss=0.9836
2026-11-15 18:06:07,266 - train.good_lr - INFO - epoch 300/500 loss=0.9360
2026-11-15 18:06:07,272 - train.good_lr - INFO - epoch 400/500 loss=0.9196
2026-11-15 18:06:07,275 - train.good_lr - INFO - epoch 500/500 loss=0.9140
2026-11-15 18:06:07,275 - train.good_lr - INFO - Run finished: w=3.054, b=1.889 (true values: 3, 2), final loss=0.9140


In [15]:
# A learning rate that's too high: the loss grows every epoch (warnings),
# then training is stopped with an error once it diverges.
train("high_lr", lr=0.1)

2026-11-15 18:06:07,278 - train.high_lr - INFO - Run started: lr=0.1, epochs=500, samples=100
2026-11-15 18:06:07,279 - train.high_lr - WARNING - Loss increased at epoch 2: 354.6899 -> 10567.1306
2026-11-15 18:06:07,280 - train.high_lr - WARNING - Loss increased at epoch 3: 10567.1306 -> 316090.2857
2026-11-15 18:06:07,280 - train.high_lr - ERROR - Training failed
Traceback (most recent call last):
  File "/var/folders/6s/tkkbfmw52494xt6qfyl36sxm0000gn/T/ipykernel_68891/2516905383.py", line 42, in train
    raise ValueError(f"Training diverged at epoch {epoch}: loss={loss:.3g}")
ValueError: Training diverged at epoch 4: loss=9.46e+06


In [16]:
# The file for the good run also has every epoch at DEBUG level
with open(os.path.join(LOG_DIR, 'train_good_lr.log')) as f:
    lines = f.readlines()
print(f"{len(lines)} lines in train_good_lr.log. First 5:")
print(''.join(lines[:5]))

507 lines in train_good_lr.log. First 5:
2026-11-15 18:06:07,249 - train.good_lr - INFO - Run started: lr=0.01, epochs=500, samples=100
2026-11-15 18:06:07,250 - train.good_lr - DEBUG - epoch=1 loss=354.6899 w=0.0000 b=0.0000
2026-11-15 18:06:07,250 - train.good_lr - DEBUG - epoch=2 loss=45.5304 w=2.1119 b=0.3312
2026-11-15 18:06:07,250 - train.good_lr - DEBUG - epoch=3 loss=6.9921 w=2.8567 b=0.4533
2026-11-15 18:06:07,250 - train.good_lr - DEBUG - epoch=4 loss=2.1833 w=3.1189 b=0.5014



## Wrap up

- Use named loggers instead of the root logger so each part of an application can be identified and controlled.
- `basicConfig` only works once unless `force=True` is passed.
- Handlers control where messages go, each with its own level and format. Turning off `propagate` avoids duplicate output.
- `RotatingFileHandler` keeps log files from growing without limit.
- JSON logs are much easier for monitoring tools to search and filter.
- In ML training, logging levels separate routine progress (INFO), detail for debugging (DEBUG), early warning signs (WARNING) and failures (ERROR), and the log files keep a record of every run.